# Frozen 3 - hollow Colab smoke

Runs in Colab **without downloading decks or ColQwen weights** and without GPU
encoding. It checks the plumbing that failed before:

- repo clone + editable install + `vvrag` import
- frozen JSON hashes/counts
- CLI flags used by the real notebook
- SQLite index shape + probe-absence logic on a synthetic 500-page index
- config load
- query-cache coverage (expected 0/180 in hollow mode)
- retrieval validation on the synthetic index (keyword only)
- provenance and bundle mechanics

A PASS here does **not** replace the real GPU run. After this passes, run
`frozen3_colab.ipynb` for the real index and query cache.


## 0. GPU report (no assert)

In [ ]:
!nvidia-smi || echo 'NO GPU VISIBLE (hollow run does not need one)'
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 1. Clone and install the exact main commit

In [ ]:
%cd /content
!if [ -d altVRAG/.git ]; then git -C altVRAG fetch --depth 1 origin main && git -C altVRAG reset --hard origin/main; else git clone --depth 1 https://github.com/BiprashPandey/altVVRAG.git altVRAG; fi
%cd /content/altVRAG
import sys, subprocess
sys.path.insert(0, '/content/altVRAG/src')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
import vvrag
print('vvrag', vvrag.__version__, vvrag.__file__)


## 2. Verify frozen inputs

In [ ]:
import hashlib, json
from pathlib import Path

REPO = Path('/content/altVRAG')
SETD = REPO / 'data' / 'slidevqa_frozen3'
META = json.loads((SETD / 'frozen_meta.json').read_text())

for name, key in [
    ('gold.json', 'gold_sha256'),
    ('probes.json', 'probes_sha256'),
    ('deck_ids.json', 'deck_ids_sha256'),
    ('questions_all.json', 'questions_all_sha256'),
]:
    got = hashlib.sha256((SETD / name).read_bytes()).hexdigest()
    assert got == META[key], f'{name} hash mismatch: {got} != {META[key]}'

gold = json.loads((SETD / 'gold.json').read_text())
probes = json.loads((SETD / 'probes.json').read_text())
decks = json.loads((SETD / 'deck_ids.json').read_text())
questions = json.loads((SETD / 'questions_all.json').read_text())
assert len(gold) == 120, f'gold count {len(gold)} != 120'
assert len(probes) == 60, f'probe count {len(probes)} != 60'
assert len(decks) == 25, f'deck count {len(decks)} != 25'
assert len(questions) == 180, f'question count {len(questions)} != 180'
assert len({x['qid'] for x in gold + probes}) == 180, 'duplicate qids'
assert (REPO / 'config.frozen3.text.yaml').exists(), 'missing frozen3 config'
print('frozen inputs OK:', len(decks), 'decks |', len(gold), 'gold |',
      len(probes), 'probes')


## 3. Raw PDFs: verify if present, otherwise trust frozen metadata

In [ ]:
raw = SETD / 'raw'
pdfs = sorted(raw.glob('*.pdf')) if raw.exists() else []
if pdfs:
    import pymupdf
    pages = 0
    for p in pdfs:
        with pymupdf.open(p) as d:
            pages += d.page_count
    assert len(pdfs) == 25, f'PDF count {len(pdfs)} != 25'
    assert pages == 500, f'page count {pages} != 500'
    print('raw PDFs OK:', len(pdfs), 'PDFs |', pages, 'pages')
else:
    assert len(META['raw_sha256']) == 25, 'frozen metadata raw hash list'
    assert META['n_pages'] == 500, 'frozen metadata page count'
    print('raw PDFs absent; metadata says 25 PDFs / 500 pages (download skipped)')


## 4. Build a synthetic 500-page SQLite index (no images, no vectors)

In [ ]:
import shutil, sqlite3
from vvrag.ingest.index import (connect_sqlite, replace_sentences,
                                replace_word_boxes, set_manifest, upsert_page)
from vvrag.schemas import PageMeta, SentenceSpan, WordBox

HOLLOW = REPO / 'data' / 'index_frozen3_hollow'
if HOLLOW.exists():
    shutil.rmtree(HOLLOW)
HOLLOW.mkdir(parents=True)
con = connect_sqlite(HOLLOW / 'meta.db')
boxes, sents = [], []
n = 0
for d in range(25):
    doc_id = f'hollow-deck-{d:02d}'
    for p in range(20):
        upsert_page(con, PageMeta(doc_id=doc_id, page=p,
                                  width=1280, height=720, dpi=150))
        text = f'hollow page {n}'
        boxes.append(WordBox(doc_id=doc_id, page=p, text=text,
                             box=(10, 10, 200, 40), sent_id=0))
        sents.append(SentenceSpan(doc_id=doc_id, page=p, sent_id=0,
                                  text=text, box=(10, 10, 200, 40)))
        n += 1
replace_word_boxes(con, boxes)
replace_sentences(con, sents)
set_manifest(con, {
    'schema_version': 2,
    'embedding_model': 'fake',
    'embedding_dim': 32,
    'dpi': 150,
    'collection_full': 'pages_full',
    'collection_pooled': 'pages_pooled',
})
con.close()
(HOLLOW / 'manifest.json').write_text(json.dumps({
    'embedding_model': 'fake',
    'embedding_dim': 32,
    'dpi': 150,
    'collection_full': 'pages_full',
    'collection_pooled': 'pages_pooled',
    'sqlite_pages': n,
    'word_boxes': len(boxes),
    'sentences': len(sents),
}, indent=2))
print('synthetic index:', n, 'pages')


## 5. Index shape and probe-absence logic

In [ ]:
con = sqlite3.connect(HOLLOW / 'meta.db')
counts = {t: con.execute(f'select count(*) from {t}').fetchone()[0]
          for t in ('pages', 'word_boxes', 'sentences')}
text = ' '.join(r[0] or '' for r in con.execute('select text from word_boxes'))
text += ' ' + ' '.join(r[0] or '' for r in con.execute('select text from sentences'))
con.close()
assert counts['pages'] == 500, counts
assert counts['word_boxes'] > 0 and counts['sentences'] > 0, counts
from vvrag.reader.claims import normalize_answer
norm = normalize_answer(text)
bad = []
for p in probes:
    for s in p.get('check_absent', []):
        if normalize_answer(s) and normalize_answer(s) in norm:
            bad.append((p['qid'], s))
assert not bad, bad[:5]
print('index shape OK:', counts, '| probes', len(probes),
      '| present checks', len(bad))


## 6. Frozen config load

In [ ]:
import os
os.environ['VVRAG_CONFIG'] = 'config.frozen3.text.yaml'
from vvrag.config import load_config
cfg = load_config()
assert cfg.reader.provider == 'gemini', cfg.reader.provider
assert cfg.reader.model == 'gemini-3.5-flash-lite', cfg.reader.model
assert cfg.verify.examiner_evidence == 'text', cfg.verify.examiner_evidence
assert cfg.retriever.query_vectors_dir == 'data/query_cache_frozen3', \
    cfg.retriever.query_vectors_dir
print('config OK:', cfg.reader.provider, cfg.reader.model,
      '| examiner', cfg.verify.examiner_evidence)


## 7. CLI flags used by the real notebook

In [ ]:
CHECKS = [
    ([sys.executable, 'scripts/prepare_slidevqa.py', '--help'],
     ['--decks-file', '--gold-out', '--max-rows', '--n-decks', '--n-qa']),
    ([sys.executable, '-m', 'vvrag.ingest.run_ingest', '--help'],
     ['--input', '--out', '--embedder', '--model-id', '--device', '--dtype',
      '--dpi', '--upscale', '--qdrant-path', '--batch-size']),
    ([sys.executable, 'scripts/precompute_query_vectors.py', '--help'],
     ['--gold', '--out', '--model', '--device', '--check']),
    ([sys.executable, 'scripts/validate_retrieval.py', '--help'],
     ['--index', '--gold', '--query-cache', '--out']),
]
for cmd, flags in CHECKS:
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    text = (proc.stdout or '') + (proc.stderr or '')
    missing = [f for f in flags if f not in text]
    assert proc.returncode == 0 and not missing, (cmd, missing, text[-300:])
print('CLI OK:', len(CHECKS), 'commands')


## 8. Query-cache coverage (hollow expectation: 0/180)

In [ ]:
from vvrag.retrieve.query_vectors import QueryVectorStore
store = QueryVectorStore('data/query_cache_frozen3')
missing_q = [q['question'] for q in questions if store.get(q['question']) is None]
print(f'query cache: {len(questions) - len(missing_q)}/{len(questions)} present')
print('hollow expectation: 0/180; the real run must assert 180/180')


## 9. Retrieval validation on the synthetic index (keyword only)

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    cache = Path(tmp) / 'cache'
    cache.mkdir()
    out = Path(tmp) / 'retrieval'
    proc = subprocess.run([
        sys.executable, 'scripts/validate_retrieval.py',
        '--index', str(HOLLOW),
        '--gold', str(SETD / 'gold.json'),
        '--query-cache', str(cache),
        '--out', str(out),
    ], capture_output=True, text=True, timeout=300)
    tail = [line for line in (proc.stdout or '').splitlines() if line.strip()][-8:]
    print('\n'.join(tail))
    assert proc.returncode == 0, (proc.stderr or '')[-1000:]
print('retrieval CLI OK')


## 10. Provenance dry run

In [ ]:
import datetime
gpu = (torch.cuda.get_device_name(0) if torch.cuda.is_available()
       else 'CPU (hollow)')
prov = {
    'created_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'repo': 'https://github.com/BiprashPandey/altVVRAG',
    'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'],
                                      cwd=REPO, text=True).strip(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': gpu,
    'colqwen_model': 'vidore/colqwen2-v1.0',
    'index_manifest': json.loads((HOLLOW / 'manifest.json').read_text()),
    'index_pages': counts['pages'],
    'index_word_boxes': counts['word_boxes'],
    'index_sentences': counts['sentences'],
    'n_frozen_decks': len(decks),
    'n_frozen_pages': META['n_pages'],
    'n_gold': len(gold),
    'n_probes': len(probes),
    'n_questions': len(questions),
    'n_query_vectors': len(questions) - len(missing_q),
}
prov_path = REPO / 'data' / 'hollow_provenance.json'
prov_path.write_text(json.dumps(prov, indent=2))
assert prov['n_gold'] == 120 and prov['n_probes'] == 60
assert prov['n_questions'] == 180
print('provenance OK:', prov_path)


## 11. Bundle mechanics (small, synthetic)

In [ ]:
import zipfile
bundle = Path('/content/frozen3_hollow_bundle.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(HOLLOW / 'meta.db', 'data/index_frozen3_hollow/meta.db')
    z.write(HOLLOW / 'manifest.json', 'data/index_frozen3_hollow/manifest.json')
    z.write(prov_path, 'results/colab_frozen3_provenance.json')
print('bundle:', bundle, bundle.stat().st_size, 'bytes')


## 12. Result

In [ ]:
print('HOLLOW COLAB PASS')
print('No decks were downloaded, no ColQwen weights loaded, no index built.')
print('Run notebooks/frozen3_colab.ipynb for the real GPU artifact.')
